## Embedding pipeline

In [2]:
from sentence_transformers import SentenceTransformer
import numpy as np
import pandas as pd
from google.cloud import bigquery
import faiss
import os

In [3]:
from google.cloud import bigquery

client = bigquery.Client()
project_id = "linear-theater-436300-r9"
dataset_id = "ecommerce_pipeline"

# Load only columns needed for embeddings
query = f"""
SELECT item_id, title, description
FROM `{project_id}.{dataset_id}.dim_product`
"""

items_df = client.query(query).to_dataframe()

print("Loaded items_df:", items_df.shape)
items_df.head()


/home/niranjanrao07/cod-multiagent-ecommerce/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:1994: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Loaded items_df: (1048018, 3)


,item_id,title,description
0,B08RYSJ3KJ,M6-1,"0 x 45mm Pan Head Phillips Machine Screws, 18-..."
1,B0BL41F5LX,6 oz,"Clear Cheese Shaker with White Lid, Server For..."
2,B004L2KFAS,John,From Publishers Weekly With plenty of imaginat...
3,1726877914,Write,Journal | White on Black Design. .
4,B011H55S4Y,Guile,Review The plot is invigorating and exciting.....


In [4]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(model_name)

print("Embedding model loaded:", model_name)

Embedding model loaded: sentence-transformers/all-MiniLM-L6-v2


In [5]:
# Combine title + description into a single text field
items_df["text_for_embedding"] = (
    items_df["title"].astype(str).str.strip() + ". " +
    items_df["description"].astype(str).str.strip()
)

# Drop rows with extremely short or empty text
items_df = items_df[items_df["text_for_embedding"].str.len() > 20]

# Reset index after filtering
items_df = items_df.reset_index(drop=True)

print("Prepared items for embedding:", items_df.shape)
items_df[["item_id", "text_for_embedding"]].head()


Prepared items for embedding: (1039122, 4)


,item_id,text_for_embedding
0,B08RYSJ3KJ,M6-1. 0 x 45mm Pan Head Phillips Machine Screw...
1,B0BL41F5LX,"6 oz. Clear Cheese Shaker with White Lid, Serv..."
2,B004L2KFAS,John. From Publishers Weekly With plenty of im...
3,1726877914,Write. Journal | White on Black Design. .
4,B011H55S4Y,Guile. Review The plot is invigorating and exc...


In [6]:
import numpy as np

# Sample 50k items for prototype retrieval
sample_size = 50000
items_sample = items_df.sample(n=sample_size, random_state=42).reset_index(drop=True)

print("Embedding sample size:", items_sample.shape)

batch_size = 2000
num_items = len(items_sample)

embeddings = []

for i in range(0, num_items, batch_size):
    batch_texts = items_sample["text_for_embedding"].iloc[i:i+batch_size].tolist()
    batch_embeddings = embedder.encode(batch_texts, show_progress_bar=True)
    embeddings.append(batch_embeddings)

embeddings = np.vstack(embeddings)

print("Final embedding array shape:", embeddings.shape)

# Save locally
np.save("data/product_embeddings_50k.npy", embeddings)
items_sample.to_csv("data/products_50k.csv", index=False)

print("Saved embeddings and sampled metadata.")


Embedding sample size: (50000, 4)


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Final embedding array shape: (50000, 384)
Saved embeddings and sampled metadata.


In [7]:
import faiss
import numpy as np

# Load embeddings and sampled metadata
embeddings = np.load("data/product_embeddings_50k.npy")
products_sample = pd.read_csv("data/products_50k.csv")

# Normalize embeddings (helps FAISS performance)
embeddings = embeddings.astype("float32")
faiss.normalize_L2(embeddings)

# Build index
d = embeddings.shape[1]  # embedding dimension
index = faiss.IndexFlatIP(d)  # cosine similarity via inner product

# Add vectors
index.add(embeddings)

print("FAISS index created.")
print("Index size:", index.ntotal)


FAISS index created.
Index size: 50000


In [8]:
# Sample query (you can change this freely)
query_text = "I need a durable water bottle suitable for hiking."

# Encode query
query_vec = embedder.encode([query_text], convert_to_numpy=True)
query_vec = query_vec.astype("float32")
faiss.normalize_L2(query_vec)

# Search FAISS index
k = 5  # top-5 results
distances, indices = index.search(query_vec, k)

print("Top-k indices:", indices[0])
print("Distances:", distances[0])

# Show results
results = products_sample.iloc[indices[0]][["item_id", "title", "description"]]
results


Top-k indices: [ 5067 44868 39922  4836 19002]
Distances: [0.64696234 0.634983   0.62315476 0.6093646  0.6025025 ]


,item_id,title,description
5067,B09JBSZRK5,sharpro 32 oz Leakproof BPA Free Drinking Spor...,Yellow/Blue Gradient). STAY HYDRATED ALL THE T...
44868,B07BKPX8YJ,Simple Modern 12oz Bolt Sports Water Bottle - ...,The Simple Modern Bolt Water Bottle is made fr...
39922,B08SQM49BJ,"Sports Water Bottle with Straw 22oz, Vacuum In...","Stainless Steel Water Bottle with Straw, Reusa..."
4836,B07L4KNTMT,OYATON Water Bottle Holder for 12 oz - 14 oz B...,Water Carrier With Adjustable Shoulder Strap A...
19002,B094Q69QHZ,Polar Bottle Sport Insulated Water Bottle - Le...,Reinvented to be your go-to sports bottle that...


In [ ]:
import faiss
import pickle

# Save FAISS index
faiss.write_index(index, "data/faiss_index_50k.bin")
print("Saved FAISS index to data/faiss_index_50k.bin")

# Save metadata again to be explicit
products_sample.to_csv("data/products_50k.csv", index=False)
print("Saved products_50k.csv")


Saved FAISS index to data/faiss_index_50k.bin
Saved products_50k.csv


: 